# Phase 3: Policy Evaluation

Goal: evaluate whether uplift scores improve targeting decisions versus random targeting.

Phase 2 produced user-level uplift scores. Phase 3 asks the business question:

`If we can only treat a fixed share of users, does ranking by predicted uplift capture more incremental conversions than random targeting?`

This is the part that turns modeling into decision science.

## 1. Setup

In [1]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")
sns.set_theme(style="whitegrid", context="notebook")

RANDOM_STATE = 42

In [2]:
cwd = Path.cwd()
PROJECT_ROOT = cwd if (cwd / "data").exists() else cwd.parent
SCORES_PATH = PROJECT_ROOT / "data" / "processed" / "uplift_scores_sample.parquet"

assert SCORES_PATH.exists(), f"Missing scored data. Run 02_uplift_modeling.ipynb first: {SCORES_PATH}"
SCORES_PATH

WindowsPath('c:/Users/roder/OneDrive/Desktop/Causal Project/causal-impact-estimation/data/processed/uplift_scores_sample.parquet')

## 2. Load Scored Test Set

This file contains the held-out test users from Phase 2 plus predicted uplift scores from the S- and T-learners.

In [9]:
df = pd.read_parquet(SCORES_PATH)
df.shape

df.head(n=5)

,treatment,conversion,visit,exposure,f0,f1,f2,f3,f4,f5,f6,f7,f8,f9,f10,f11,s_pred_treated,s_pred_control,s_uplift,t_pred_treated,t_pred_control,t_uplift
0,1,0,0,1,14.3667,10.0597,8.2144,3.3598,10.2805,4.1155,-2.4111,4.8338,3.9719,13.1901,5.3004,-0.1687,0.0003,0.0003,0.0000,0.0001,0.0011,-0.0010
1,0,0,0,0,24.3978,10.0597,8.2144,4.6799,10.2805,4.1155,-2.4111,4.8338,3.9719,13.1901,5.3004,-0.1687,0.0001,0.0001,0.0000,0.0001,0.0011,-0.0010
2,0,0,0,0,24.5717,10.0597,8.2144,4.6799,10.2805,4.1155,-3.2821,4.8338,3.9719,13.1901,5.3004,-0.1687,0.0001,0.0001,0.0000,0.0001,0.0011,-0.0010
3,1,0,0,0,24.3681,10.0597,8.2144,4.6799,10.2805,4.1155,-5.1167,4.8338,3.9719,13.1901,5.3004,-0.1687,0.0001,0.0001,0.0000,0.0001,0.0011,-0.0010
4,1,0,0,0,24.3531,10.0597,8.2144,4.6799,10.2805,4.1155,-2.4111,4.8338,3.9719,13.1901,5.3004,-0.1687,0.0001,0.0001,0.0000,0.0001,0.0011,-0.0010


In [4]:
treatment_col = "treatment"
outcome_col = "conversion"
score_cols = ["s_uplift", "t_uplift"]

df.groupby(treatment_col)[outcome_col].agg(users="size", conversion_rate="mean")

,users,conversion_rate
treatment,,
0,75000,0.0019
1,75000,0.0029


In [5]:
df[score_cols].describe(percentiles=[0.01, 0.05, 0.10, 0.25, 0.50, 0.75, 0.90, 0.95, 0.99])

,s_uplift,t_uplift
count,"150,000.0000","150,000.0000"
mean,0.0007,0.0002
std,0.0111,0.0305
min,-0.2354,-0.9977
1%,-0.0007,-0.0021
5%,-0.0000,-0.0010
10%,-0.0000,-0.0010
25%,0.0000,-0.0010
50%,0.0000,-0.0010
75%,0.0000,-0.0008


## 3. How Uplift Policy Evaluation Works

For a ranked list of users, estimated incremental conversions up to a cutoff are:

`treated conversions - control conversions * (treated users / control users)`

Why scale control conversions? Because each prefix of the ranked list may contain different numbers of treated and control users. The scaling estimates how many conversions we would have expected among treated users if they behaved like comparable control users in the same ranked prefix.

A useful uplift model should accumulate incremental conversions faster than random targeting.

## 4. Metric Helpers

In [10]:
def uplift_curve(data, score_col, treatment_col="treatment", outcome_col="conversion"):
    """Return cumulative uplift curve after sorting users by score descending."""
    ranked = data.sort_values(score_col, ascending=False).reset_index(drop=True).copy()
    ranked["rank"] = np.arange(1, len(ranked) + 1)
    ranked["population_share"] = ranked["rank"] / len(ranked)

    ranked["treated_user"] = (ranked[treatment_col] == 1).astype(int)
    ranked["control_user"] = (ranked[treatment_col] == 0).astype(int)
    ranked["treated_conversion"] = ranked["treated_user"] * ranked[outcome_col]
    ranked["control_conversion"] = ranked["control_user"] * ranked[outcome_col]

    ranked["cum_treated_users"] = ranked["treated_user"].cumsum()
    ranked["cum_control_users"] = ranked["control_user"].cumsum()
    ranked["cum_treated_conversions"] = ranked["treated_conversion"].cumsum()
    ranked["cum_control_conversions"] = ranked["control_conversion"].cumsum()

    control_scale = ranked["cum_treated_users"] / ranked["cum_control_users"].replace(0, np.nan)
    ranked["cum_incremental_conversions"] = (
        ranked["cum_treated_conversions"]
        - ranked["cum_control_conversions"] * control_scale
    )
    ranked["cum_incremental_conversions"] = ranked["cum_incremental_conversions"].fillna(0)

    total_incremental = ranked["cum_incremental_conversions"].iloc[-1]
    ranked["gain_share"] = ranked["cum_incremental_conversions"] / total_incremental if total_incremental != 0 else np.nan

    return ranked


def random_baseline(curve):
    total_incremental = curve["cum_incremental_conversions"].iloc[-1]
    return curve["population_share"] * total_incremental


def auuc(curve):
    return np.trapezoid(
        y=curve["cum_incremental_conversions"],
        x=curve["population_share"],
    )


def qini_auc(curve):
    return np.trapezoid(
        y=curve["cum_incremental_conversions"] - random_baseline(curve),
        x=curve["population_share"],
    )


def top_k_summary(curve, budgets=(0.10, 0.20, 0.30, 0.50)):
    rows = []
    total_incremental = curve["cum_incremental_conversions"].iloc[-1]

    for budget in budgets:
        idx = (curve["population_share"] - budget).abs().idxmin()
        incremental = curve.loc[idx, "cum_incremental_conversions"]
        random_incremental = budget * total_incremental
        incremental_vs_random = incremental - random_incremental
        lift_vs_random = incremental / random_incremental - 1 if random_incremental != 0 else np.nan

        rows.append({
            "budget_share": budget,
            "users_targeted": int(curve.loc[idx, "rank"]),
            "incremental_conversions": incremental,
            "random_incremental_conversions": random_incremental,
            "incremental_conversions_vs_random": incremental_vs_random,
            "lift_vs_random": lift_vs_random,
            "share_of_total_incremental": incremental / total_incremental if total_incremental != 0 else np.nan,
        })

    return pd.DataFrame(rows)

## 5. Build Curves for Each Learner

In [11]:
curves = {score_col: uplift_curve(df, score_col) for score_col in score_cols}

metric_summary = pd.DataFrame([
    {
        "score": score_col,
        "total_incremental_conversions": curve["cum_incremental_conversions"].iloc[-1],
        "auuc": auuc(curve),
        "qini_auc_vs_random": qini_auc(curve),
    }
    for score_col, curve in curves.items()
]).sort_values("qini_auc_vs_random", ascending=False)

metric_summary

AttributeError: module 'numpy' has no attribute 'trapz'

Interpretation:

- Higher AUUC means the model accumulates more incremental conversions earlier in the ranked list.
- Higher Qini AUC versus random means the targeting policy beats random allocation by more area under the curve.
- A negative Qini AUC means the ranking is worse than random over much of the curve.

## 6. Cumulative Incremental Gain Curves

In [ ]:
plt.figure(figsize=(10, 6))

for score_col, curve in curves.items():
    plt.plot(
        curve["population_share"],
        curve["cum_incremental_conversions"],
        label=score_col,
    )

first_curve = next(iter(curves.values()))
plt.plot(
    first_curve["population_share"],
    random_baseline(first_curve),
    linestyle="--",
    color="black",
    label="random baseline",
)

plt.axhline(0, color="#D62728", linestyle=":")
plt.title("Cumulative Incremental Conversions by Targeting Budget")
plt.xlabel("Share of users targeted")
plt.ylabel("Estimated cumulative incremental conversions")
plt.legend()
plt.tight_layout()

## 7. Qini Curves

The Qini curve subtracts the random targeting baseline. Values above zero mean the model is beating random targeting at that budget level.

In [ ]:
plt.figure(figsize=(10, 6))

for score_col, curve in curves.items():
    plt.plot(
        curve["population_share"],
        curve["cum_incremental_conversions"] - random_baseline(curve),
        label=score_col,
    )

plt.axhline(0, color="black", linestyle="--", label="random baseline")
plt.title("Qini Curve: Incremental Conversions Above Random")
plt.xlabel("Share of users targeted")
plt.ylabel("Incremental conversions above random")
plt.legend()
plt.tight_layout()

## 8. Top-K Targeting Summary

This table answers the resume/business question: at a fixed treatment budget, how many more incremental conversions does uplift targeting capture than random targeting?

In [ ]:
top_k_tables = []

for score_col, curve in curves.items():
    table = top_k_summary(curve)
    table.insert(0, "score", score_col)
    top_k_tables.append(table)

top_k = pd.concat(top_k_tables, ignore_index=True)
top_k.sort_values(["budget_share", "lift_vs_random"], ascending=[True, False])

If `lift_vs_random = 0.50`, that means the uplift policy captured 50% more incremental conversions than random targeting at the same treatment budget.

## 9. Recommended Budget Cutoff

A policy is not just a model. A policy says who gets treated. Here we select the model-budget pair with the strongest incremental conversions versus random among the tested budgets.

In [ ]:
policy_candidates = top_k.dropna(subset=["lift_vs_random"]).copy()
policy_candidates = policy_candidates[policy_candidates["incremental_conversions"] > 0]

best_policy = policy_candidates.sort_values(
    ["lift_vs_random", "incremental_conversions"],
    ascending=False,
).head(1)

best_policy

This recommendation is still offline. In a real product setting, we would validate it with a follow-up randomized experiment where the uplift policy competes against random allocation or the current production targeting rule.

## 10. Phase 3 Readout

Fill this in after running the notebook:

- The best-performing score was `[score]` based on Qini AUC versus random.
- At a `[K%]` treatment budget, the uplift policy captured `[X]` estimated incremental conversions versus `[Y]` under random targeting.
- This represents `[Z%]` more incremental conversions than random targeting at the same budget.
- The cumulative gain/Qini curves show `[clear improvement / mixed results / weak evidence]`.
- Recommendation: use the current uplift scores as `[a policy candidate / a diagnostic baseline only]`, and validate with a follow-up randomized policy test.

Caveat: these results are based on a sampled offline test set. The final project should rerun the strongest learner on a larger sample or the full dataset before reporting final resume numbers.